# Integrating with LangChain

LangChain's `ChatOpenAI` talks to OpenAI through the `openai` SDK, so `immune.init()` covers it with no callbacks,
wrappers or changes to your chains. This guide shows where Immune sits in common LangChain applications.

| You have | Section |
| --- | --- |
| A prompt → model → parser chain | [1](#1.-A-support-chain) |
| Retrieval-augmented generation (RAG) | [2](#2.-RAG-over-your-help-center) |
| A way to test RAG against a poisoned document | [3](#3.-Rehearse-a-poisoned-document) |
| Tool calling with `bind_tools` | [4](#4.-Tool-calling-with-bind_tools) |
| Streaming and async chains | [5](#5.-Streaming-and-async) |

**Requirements:** `pip install immune-ai langchain langchain-openai numpy` (numpy is only for the in-memory vector
store in the RAG example), with `OPENAI_API_KEY` and `TYPESAFE_API_KEY` set. Other chat models built on the OpenAI, Anthropic or Google SDKs (`ChatAnthropic`, `ChatGoogleGenerativeAI`)
are covered the same way.

```
 your chain ──► ChatOpenAI ──► openai SDK ──► HTTP transport ──► OpenAI
                                                   ▲
                                        immune.init() hooks in here,
                                        below every framework
```

## 1. A support chain

**Before.**

In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are the support assistant for Pinecrest Outfitters, an online camping and hiking store."),
        ("human", "{question}"),
    ]
)
chain = prompt | ChatOpenAI(model="gpt-5.4-mini") | StrOutputParser()

print(chain.invoke({"question": "What should I look for in a three-season sleeping bag?"}))

For a **three-season sleeping bag** (spring, summer, fall), look for these key features:

- **Temperature rating:** Aim for a bag rated around **20–35°F (-6 to 2°C)**, depending on how cold you sleep and where you camp.
- **Insulation type:**
  - **Down:** Lighter, more compressible, great for backpacking, but loses warmth if wet unless treated.
  - **Synthetic:** Warmer when damp and usually less expensive, but bulkier and heavier.
- **Shape:** 
  - **Mummy bags** are the warmest and most packable.
  - **Rectangular bags** are roomier but less efficient in cooler weather.
- **Weight and pack size:** Important if you’re carrying it long distances.
- **Fit:** Make sure it’s snug enough to keep warmth in, but not so tight that you can’t move.
- **Features:** Draft collar, draft tube along the zipper, hood, and a good zipper all help retain heat.
- **Shell material and water resistance:** A durable, water-resistant outer fabric is helpful for damp conditions.
- **Gender/size options:** Ch

**After.** Add `immune.init()` at startup and name the site. The chain is unchanged.

```diff
+ import immune
  from langchain_openai import ChatOpenAI
  ...
+ immune.init()
  chain = prompt | ChatOpenAI(model="gpt-5.4-mini") | StrOutputParser()
+ with immune.site("support-chat"):
      answer = chain.invoke(...)
```

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import immune

immune.init()
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are the support assistant for Pinecrest Outfitters, an online camping and hiking store."),
        ("human", "{question}"),
    ]
)
chain = prompt | ChatOpenAI(model="gpt-5.4-mini") | StrOutputParser()

with immune.site("support-chat"):
    answer = chain.invoke({"question": "What should I look for in a three-season sleeping bag?"})
print(answer)

verdict = immune.verdict()  # the latest verdict recorded inside the site block
print("\nverdict:", verdict.action, "| site:", verdict.site)

For a **three-season sleeping bag** (spring, summer, fall), look for these key things:

- **Temperature rating:** Choose a bag rated for the coldest conditions you expect. A good starting point is often around **20–35°F (-6 to 2°C)**, depending on where and how you sleep.
- **Insulation type:**
  - **Down:** Lighter, more packable, and warmer for the weight; best if you can keep it dry.
  - **Synthetic:** Better in damp conditions and usually less expensive, but bulkier and heavier.
- **Fit and shape:**
  - **Mummy bags** are warmer and lighter.
  - **Rectangular or semi-rectangular bags** give more room but may be less efficient in cooler weather.
- **Weight and pack size:** Important if you’re backpacking. Lighter and more compressible bags are easier to carry.
- **Comfort features:** Draft collar, hood, zipper draft tube, and a well-designed footbox help retain warmth.
- **Shell fabric and durability:** Look for a shell that’s comfortable, durable, and water-resistant if possible.
-

> **Where to read the verdict.** A parser returns plain text, so there is no response object to pass to
> `immune.verdict(...)`. LangChain also runs each step in a copy of the context, so read `immune.verdict()` *after*
> the `with immune.site(...)` block: the block hands back the latest verdict recorded inside it. In concurrent code,
> keep the model's `AIMessage` and call `immune.verdict(message)` (section 4), or collect verdicts with
> `immune.on_verdict(...)`.

## 2. RAG over your help center

**Before.** A standard retrieval chain: help-center articles in a vector store, the top matches pasted into the
prompt as context.

In [3]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

articles = [
    Document(
        "Returns: unused items can be returned within 30 days with a receipt. Worn footwear can't be returned.",
        metadata={"source": "help/returns"},
    ),
    Document(
        "Shipping: free on orders over $75. Standard delivery takes 3-5 business days; express takes 1-2.",
        metadata={"source": "help/shipping"},
    ),
    Document(
        "Warranty: tents and packs have a 2-year warranty against manufacturing defects, such as broken poles.",
        metadata={"source": "help/warranty"},
    ),
]
retriever = InMemoryVectorStore.from_documents(articles, OpenAIEmbeddings()).as_retriever(search_kwargs={"k": 2})


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer questions for Pinecrest Outfitters customers using only the context provided."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-5.4-mini")
    | StrOutputParser()
)
print(rag_chain.invoke("My tent pole snapped after a month. Am I covered?"))

Yes — tents have a 2-year warranty against manufacturing defects, and a broken pole is listed as an example. If the pole snapped due to a manufacturing defect, it should be covered.


**After.** Retrieved text is *data*: it should inform the answer, never give instructions. Tell Immune which part of
the prompt is retrieved by wrapping it in `immune.untrusted(...)`. That one change in `format_docs` is the whole
integration.

```diff
  def format_docs(docs):
-     return "\n\n".join(doc.page_content for doc in docs)
+     return "\n\n".join(immune.untrusted(doc.page_content, source=doc.metadata["source"]) for doc in docs)
```

The markers are removed before the model sees the text; the model gets exactly the same prompt as before.

In [4]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

import immune

immune.init()
articles = [
    Document(
        "Returns: unused items can be returned within 30 days with a receipt. Worn footwear can't be returned.",
        metadata={"source": "help/returns"},
    ),
    Document(
        "Shipping: free on orders over $75. Standard delivery takes 3-5 business days; express takes 1-2.",
        metadata={"source": "help/shipping"},
    ),
    Document(
        "Warranty: tents and packs have a 2-year warranty against manufacturing defects, such as broken poles.",
        metadata={"source": "help/warranty"},
    ),
]
retriever = InMemoryVectorStore.from_documents(articles, OpenAIEmbeddings()).as_retriever(search_kwargs={"k": 2})


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(immune.untrusted(doc.page_content, source=doc.metadata["source"]) for doc in docs)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer questions for Pinecrest Outfitters customers using only the context provided."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-5.4-mini")
    | StrOutputParser()
)
with immune.site("help-center"):
    print(rag_chain.invoke("My tent pole snapped after a month. Am I covered?"))

verdict = immune.verdict()
print("\nverdict:", verdict.action, "| taint:", verdict.taint, "| Jev requests:", verdict.sensor.calls)

Yes—if the pole snapped due to a manufacturing defect, your tent is covered under the 2-year warranty for tents and packs, which includes broken poles.

If you’d like, I can also help you figure out whether it sounds like a warranty issue or a return issue.

verdict: allow | taint: external | Jev requests: 4


**What you see:** the same answer. The verdict's `taint: external` shows Immune knows this conversation read outside
content; every retrieved document was screened as data, in parallel with the model call. Embedding calls are not LLM
chat calls, so Immune lets them pass untouched.

## 3. Rehearse a poisoned document

Immune ships a library of reconstructed, publicly reported incidents. You can run one through your own chain,
offline, to see what Immune does. Here the RAG chain above receives the document from `incident.echoleak`: an email
with hidden instructions that make the assistant put a data-carrying image link in its answer. The model's side is
the recorded reply from the incident, served by Immune's test harness, so no real model is asked to misbehave.

In [5]:
import tempfile
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import immune
from immune.testing import ImmuneHarness, MockSensor, ScenarioLibrary

incident = ScenarioLibrary().find("incident.echoleak")
turn = incident.turns[0]
harness = ImmuneHarness(
    Path(tempfile.mkdtemp()),
    sensor=MockSensor(incident.signals),  # Jev's answers, as recorded for this incident
    script=turn.reply.fake(),  # the model's recorded reply
)
model = ChatOpenAI(model="gpt-5.4-mini", api_key="rehearsal", http_client=harness.http_client(), max_retries=0)

prompt = ChatPromptTemplate.from_messages(
    [("system", incident.operator), ("human", "{question}\n\nContext:\n{context}")]
)
context = "\n\n".join(immune.untrusted(item.text, source="mailbox") for item in turn.data)
message = (prompt | model).invoke({"question": turn.user, "context": context})

print("incident:", incident.title)
print("the model's recorded reply contained a link:", "](http" in turn.reply.text)
print("what your app received:", message.content)
verdict = harness.verdict(message)
print("\nverdict:", verdict.action)
for hit in verdict.hits:
    print(f"  {hit.threat:<34} enforced={hit.enforced}  {hit.floor or ''}")
harness.close()

incident: EchoLeak: a poisoned email makes the assistant exfiltrate data through an image URL
the model's recorded reply contained a link: True
what your app received: - Q3 numbers received
[link removed]

verdict: rewrite
  data.instructions                  enforced=True  F7
  data.exfil_request                 enforced=False  
  output.exfil_link                  enforced=True  F2
  output.follows_data_instructions   enforced=False  


**What you see:** two floor rules acted.

- **F7** found the planted instructions in the email and replaced it with a notice, so in a live run the model never
  reads them.
- **F2** removed the data-carrying link from the reply, so even a model that followed the instructions leaks nothing.

The observed hits are detectors that saw the same attack but are not enforced by default. Rehearsals like this
belong in your test suite; see [Testing and CI](13_testing_and_ci.ipynb).

## 4. Tool calling with `bind_tools`

**Before.**

In [6]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def check_stock(product: str) -> str:
    """Check how many units of a product are in stock."""
    return {"ridgeline 2p": "12 in stock", "summit lite 2p": "out of stock"}.get(product.lower(), "unknown product")


model = ChatOpenAI(model="gpt-5.4-mini").bind_tools([check_stock])
message = model.invoke("Is the Summit Lite 2P in stock?")
for call in message.tool_calls:
    print(call["name"], call["args"], "->", check_stock.invoke(call["args"]))

check_stock {'product': 'Summit Lite 2P'} -> out of stock


**After.** The model's tool calls are screened before your code sees them. Because LangChain keeps the provider's
response id on the message, `immune.verdict(message)` finds the verdict, even in concurrent code.

In [7]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

import immune

immune.init()


@tool
def check_stock(product: str) -> str:
    """Check how many units of a product are in stock."""
    return {"ridgeline 2p": "12 in stock", "summit lite 2p": "out of stock"}.get(product.lower(), "unknown product")


model = ChatOpenAI(model="gpt-5.4-mini").bind_tools([check_stock])
with immune.site("stock-assistant"):
    message = model.invoke("Is the Summit Lite 2P in stock?")
for call in message.tool_calls:
    print(call["name"], call["args"], "->", check_stock.invoke(call["args"]))

print("\nverdict:", immune.verdict(message).action)

check_stock {'product': 'Summit Lite 2P'} -> out of stock

verdict: allow


For agents that loop over tool calls, see [LangGraph](05_langgraph.ipynb).

## 5. Streaming and async

`stream`, `astream`, `ainvoke` and `abatch` all go through the same SDK, so they are screened the same way.

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import immune

immune.init()
chain = (
    ChatPromptTemplate.from_messages([("system", "You are a concise hiking guide."), ("human", "{question}")])
    | ChatOpenAI(model="gpt-5.4-mini")
    | StrOutputParser()
)

with immune.site("hiking-guide"):
    for piece in chain.stream({"question": "Name three things to check before a winter hike."}):
        print(piece, end="", flush=True)
print("\n\nverdict:", immune.verdict().action)

1. **Weather forecast and 

wind chill**
2. **Trail conditions and avalanche risk**
3. **Your gear, especially layers, traction, and navigation tools**



verdict: allow


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import immune

immune.init()
chain = ChatPromptTemplate.from_messages(
    [("system", "Answer in one sentence for Pinecrest Outfitters customers."), ("human", "{question}")]
) | ChatOpenAI(model="gpt-5.4-mini")

questions = [{"question": "Do trekking poles help on descents?"}, {"question": "Is merino wool good for socks?"}]
messages = await chain.abatch(questions)  # in a script: asyncio.run(chain.abatch(questions))
for question, message in zip(questions, messages, strict=True):
    print(f"[{immune.verdict(message).action}] {question['question']}\n        {message.content}")

[allow] Do trekking poles help on descents?
        Yes—trekking poles can improve balance and reduce knee strain on descents, especially with a full pack or on steep, loose terrain.
[allow] Is merino wool good for socks?
        Yes—merino wool is excellent for socks because it’s soft, breathable, moisture-wicking, and helps keep your feet warm in cold weather and cooler when it’s warm.


## Common mistakes

| Avoid | Prefer instead | Why |
| --- | --- | --- |
| Putting retrieved context in the **system** message | The human message, wrapped in `immune.untrusted(...)` | The system message is trusted operator text |
| Adding Immune as a LangChain callback or wrapper | `immune.init()` at startup | Immune works below LangChain; nothing else is needed |
| `immune.verdict()` inside the chain's `with` block, or after `abatch` | After the `with immune.site(...)` block, or `immune.verdict(message)` | LangChain copies the context; concurrent calls each have their own verdict |

## Recap

- LangChain chains, retrievers, tools, streaming and async are covered by `immune.init()`
- In RAG, wrap retrieved text in `immune.untrusted(...)`; the model sees the same prompt
- `immune.verdict(message)` works with LangChain `AIMessage` objects
- Rehearse library incidents through your own chain with `ImmuneHarness`

Next: [LangGraph](05_langgraph.ipynb) · [Untrusted data and RAG](09_untrusted_data_and_rag.ipynb)